## Lernziel

Produkte, Kategorien und Referenzwerte mit den vorhandenen Baubook-Funktionen abrufen und tabellarisch auswerten.

**Voraussetzungen:** Eigener Baubook-API-Zugang und Internetverbindung; lokale Zugangsdaten.

Einmalig im Repository installieren: `python -m pip install -e ".[publication]"`.
Zum eigenen Ausführen das Notebook im geklonten Repository öffnen und die Zellen von oben nach unten ausführen.
Die Website zeigt gespeicherte Ausgaben; sie führt keinen Python-Code im Browser aus.

[Tutorial-Notebook herunterladen](downloads/tutorial_baubook.ipynb) · [Alle Tutorials](python.qmd)


## 1. Zugang vorbereiten

Zusätzlich wird `requests` benötigt (`python -m pip install requests`).
Lege lokal eine nicht versionierte `credentials.json` im Repository an:

```json
{"user": "DEIN_BENUTZERNAME", "api_key": "DEIN_API_SCHLUESSEL"}
```

Dieses Tutorial enthält keine gespeicherten API-Antworten. Die folgenden Zellen benötigen deinen Zugang;
sie wurden für die Website nicht gegen den Dienst ausgeführt. Zugangsdaten und Sitzungstoken nicht ausgeben.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

# Im geklonten Repository ausführen, z. B. aus website/ oder notebooks/.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/pexl").is_dir() and (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Bitte das Notebook innerhalb des peexcel-Repositorys öffnen.")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
OUTPUT = ROOT / "notebooks/local_exports/tutorials"
OUTPUT.mkdir(parents=True, exist_ok=True)
from pexl.io import baubook
ZUGANG = ROOT / "credentials.json"
benutzer, schluessel = baubook.load_credentials(ZUGANG)
session = baubook.create_baubook_session(benutzer, schluessel)

## 2. Eine kleine Produktmenge prüfen

Zuerst fünf Produkte laden. `json_normalize` wandelt die API-Antwort in eine Tabelle um;
die tatsächlichen Spalten ergeben sich aus den gelieferten Daten.

In [ ]:
antwort = baubook.get_products(session, b_size=5)
produkte = pd.json_normalize(antwort["items"])
print("Produkte in dieser Antwort:", len(produkte))
display(produkte.head())

## 3. Kategorien nachschlagen

Kategorien können Unterkategorien enthalten. Die API-Bezeichnungen bleiben erhalten;
sie werden nicht durch ein zusätzliches lokales Kategorieschema ersetzt.

In [ ]:
kategorien = baubook.get_categories(session)
display(pd.DataFrame([
    {"ID": key, "Bezeichnung": value.get("Titel", {}).get("de", "")}
    for key, value in kategorien.items()
]))

## 4. Referenzwerte und Export

Referenzwerte getrennt von Produktdatensätzen prüfen. Ein Tabellenauszug ist noch keine Zuordnung
zu Bauteilen oder eine fertige Ökobilanz. Einheiten und Gültigkeitsbereiche aus den Antwortfeldern übernehmen.

In [ ]:
referenzwerte = baubook.get_reference_values(session, "ZG")
referenzen = pd.DataFrame.from_dict(referenzwerte["items"], orient="index").reset_index(names="reference_id")
display(referenzen.head())
produkte.to_excel(OUTPUT / "tutorial_baubook_produkte.xlsx", index=False)
referenzen.to_excel(OUTPUT / "tutorial_baubook_referenzwerte.xlsx", index=False)
session.close()

## Ergebnis und Grenzen

Die beiden Excel-Dateien enthalten die abgerufenen Tabellen. Verfügbarkeit und Umfang hängen vom API-Zugang ab.
Vor einer größeren Abfrage Produktzahl und API-Antwort prüfen; das Beispiel lädt nicht automatisch den gesamten Katalog.



## Weiterarbeiten

[Zur Tutorial-Übersicht](python.qmd). Die ursprünglichen Arbeitsnotebooks bleiben erhalten: [howto_baubook_io](https://github.com/simonschaluppe/peexcel/blob/master/notebooks/howto_baubook_io.ipynb).
